In [46]:
import json
import re
import sys

import pandas as pd
from sqlalchemy import create_engine

from nhs_waiting_lists.utils.proj_paths import find_project_root

project_root = find_project_root()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [47]:

from nhs_waiting_lists.constants import proj_db_path

DB_PATH = project_root / proj_db_path / "nhs_rttwtd.db"
DATA_DIR = "./data"

conn = create_engine(f"sqlite:///{DB_PATH}")


In [54]:

from nhs_waiting_lists.utils.tables import create_outpatient_activity_table
from scripts.excel_parsing_olde.parse_nhs_data import load_data_to_database2

connection = conn.raw_connection()
create_outpatient_activity_table(connection)

In [67]:


op_pla_file = r'hosp-epis-stat-outp-pla-([0-9]{4})-([0-9]{2})-(?:data|tab(?:%20v2)?)\.(csv|xlsx?m?)'



def process_op_excel(data, year, month, suffix):
    # df = pd.read_excel(
    #     project_root / 'files' / data["path"],
    #     sheet_name="Provider Table"
    # )
    print(f"processing {data['path']} - TODO")

def process_op_file(data):
    match = re.search(op_pla_file, data["filename"])
    if not match:
        print(f"no match for {data['filename']}")
        return

    year, yr_next, suffix = match.groups()
    print(f"year: {year}, yr_next: {yr_next} suffix: {suffix}")
    if suffix == "xlsx":
        df = process_op_excel(data, year, yr_next, suffix)
    else:  #  suffix == "csv"
        df = process_op_csv(data, year, yr_next)

    if isinstance(df, pd.DataFrame):

        load_data_to_database2(df, "outpatients_activity", connection)



with open(project_root / "files/downloads_outpatient-activity.jsonl") as f:
    is_looping = True
    for line in f:
        data = json.loads(line)
        for foo in data["files"]:
            print(f"file is {project_root / 'files' / foo['path']}")
            process_op_file(foo)


file is /home/tomhodder/Sync/projects/data/nhs_england_data/files/outpatient-activity/hosp-epis-stat-outp-pla-2020-21-tab.xlsx
year: 2020, yr_next: 21 suffix: xlsx
processing outpatient-activity/hosp-epis-stat-outp-pla-2020-21-tab.xlsx - TODO
file is /home/tomhodder/Sync/projects/data/nhs_england_data/files/outpatient-activity/hosp-epis-stat-outp-pla-2021-22-tab.xlsx
year: 2021, yr_next: 22 suffix: xlsx
processing outpatient-activity/hosp-epis-stat-outp-pla-2021-22-tab.xlsx - TODO
file is /home/tomhodder/Sync/projects/data/nhs_england_data/files/outpatient-activity/hosp-epis-stat-outp-pla-2022-23-data.csv
year: 2022, yr_next: 23 suffix: csv
reporting_period      object
geography_level       object
organisation_code     object
measure_type          object
measure               object
measure_value        float64
dtype: object
      reporting_period geography_level organisation_code  \
53328          2022-23        Provider             RYR-X   
53329          2022-23        Provider     